# 18 - Train Offline DQN on CartPole

This notebook trains on the CartPole streams from `17_collect_cartpole.ipynb`. The loop matches `02_train_offline_dqn.ipynb`: load stores, pack tasks with a `Tokenizer`, run a pretrained backbone, and update a `RegressionHead` with `DqnObjective`.

The difference is the observation. CartPole stores a length-4 float vector. A text field with `input_index` reads one element and renders it as a scalar. Four fields, indices `0` through `3`, emit in list order:

`cart_position, cart_velocity, pole_angle, pole_angular_velocity`

Each element uses `,{field:.3f}`, so the four numbers follow the action as a comma-separated list. Each piece is its own tokenize call, so a number's tokens do not merge with its neighbors.

Actions stay discrete (`0` or `1`). The augmenter permutes those ids per task, the same way `02` permutes FrozenLake actions, so an id has no fixed meaning across tasks. There is no observation permutation: the vector is continuous, not a vocab id.

This is a short usage example, not a full experiment.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    to_device,
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective, boundary_discount
from mouse_core.models import Model, model_polyak, push_model_to_hub
from mouse_core.models.backbone import TransformerBackbone
from mouse_core.models.heads import RegressionHead


DATASET_ID = "mouse-example-cartpole-dataset"              # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-cartpole-model-offline"          # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-cartpole-tokenizer-offline"  # Hugging Face tokenizer repo (separate from MODEL_ID)
PRETRAINED = "Qwen/Qwen3-0.6B"                             # HF checkpoint for Tokenizer and backbone
MAX_ACTIONS = 2                                            # CartPole discrete actions
MAX_EPISODE_STEPS = 50                                     # max steps per episode (matches collect)
MAX_TASK_EPISODES = 4                                      # max episodes per task (matches collect)
TOKEN_BUDGET = 16384                                       # packed tokens; whole tasks are added until the next one would not fit
NUM_CYCLES = 2                                             # outer train cycles (print cadence)
TRAIN_STEPS = 50                                           # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                                  # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_BACKBONE = 0.01                                 # delayed backbone interpolation

LR_BACKBONE = 1e-5                                         # AdamW lr for the backbone
LR_HEADS = 1e-5                                            # AdamW lr for the heads


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` rebuilds the `Datastore` streams saved by `17`. Each store is one ordered environment. `observation` on every row is a length-4 vector.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train", force_download=True)


## Data pipeline

`sample_field="task_index"` makes each packed group one task. `batch_size=1` fills that budget once. `token_budget` counts packed tokens, including the group-start prefix. A task that does not fit is left out. A first task that alone is over the budget raises. `MAX_EPISODE_STEPS` and `MAX_TASK_EPISODES` in `17` keep one task under `TOKEN_BUDGET`.

`input_index` is the element number inside `observation`:

| Index | Element |
| --- | --- |
| 0 | cart position |
| 1 | cart velocity |
| 2 | pole angle (radians) |
| 3 | pole angular velocity |

`{field:.3f}` prints three digits. Reward `0.0` and done code `0` use `when` callables so a zero value emits nothing. A reset frame has reward `0` and emits no `,r=`. A step that scores `1` emits `,r=1`. `episode_index` emits `,e={field}` when `step_index` is `0` or at group start. `task_done` is an objective column only. The const newline is `head_output`: the Q readout.

`type="linear"` on an `Augmenter` field can rescale the vector before these indices. This notebook does not: the raw components are rendered as collected.


In [ ]:
def when_episode_done_nonzero(ctx):
    return "episode_done" in ctx and ctx["episode_done"] != 0

def when_group_start(ctx):
    return bool(ctx["group_start"])

def when_reward_nonzero(ctx):
    return "reward" in ctx and ctx["reward"] != 0.0

def when_step_index_zero_or_group_start(ctx):
    return (ctx.get("step_index") == 0) | bool(ctx["group_start"])


group_prefix = (
    "Your job is to predict the future sum of rewards in CartPole. "
    "Balance a pole on a cart. Each step that the pole stays up scores 1. "
    "The episode ends when the pole falls or the cart leaves the track. "
    f"You have {MAX_TASK_EPISODES} episodes to solve the task. "
    "Action ids may be remapped.\n"
    "Predict when you see a new line. Step format: "
    "action,cart_position,cart_velocity,pole_angle,pole_angular_velocity"
    "[,r=reward][,d=done][,e=episode].\n"
)


augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "text",
            "output_field": "group_start",
            "format": group_prefix,
            "when": when_group_start,
        },
        {
            "type": "text",
            "input_field": "action",
            "format": "{field}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "input_index": 0,
            "format": ",{field:.3f}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "input_index": 1,
            "format": ",{field:.3f}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "input_index": 2,
            "format": ",{field:.3f}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "input_index": 3,
            "format": ",{field:.3f}",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": ",r={field:g}",
            "when": when_reward_nonzero,
        },
        {
            "type": "text",
            "input_field": "episode_done",
            "format": ",d={field}",
            "when": when_episode_done_nonzero,
        },
        {
            "type": "text",
            "input_field": "episode_index",
            "format": ",e={field}",
            "when": when_step_index_zero_or_group_start,
        },
        {
            "type": "text",
            "output_field": "value",
            "format": "\n",
            "max_tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {"input_field": "action"},
        {"input_field": "reward"},
        {"input_field": "episode_done"},
        {"input_field": "task_done"},
    ],
    pretrained=PRETRAINED,
)

train_transform = compose(stages=(augmenter, tokenizer))

loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    batch_size=1,
    sample_field="task_index",
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

Same pieces as `02`. `TransformerBackbone(pretrained=...)` loads `embed_tokens` and looks up the packed text ids. `train_kernel`, `decode_kernel`, and `dtype` are how this machine runs the model, so they are not saved; `load_model` asks for them again. `dtype=torch.float32` trains the backbone in fp32. `use_norm=True` keeps the final RMSNorm.

`RegressionHead` predicts one value per discrete action (`MAX_ACTIONS` is `2`). `action_source="action_value"` is the head `get_action` reads.

### What packed steps look like

`*\n*` is the head-output token. The head reads Q from it.

```text
task=0
Your job is to predict the future sum of rewards in CartPole. ...
Predict when you see a new line. Step format: action,cart_position,cart_velocity,pole_angle,pole_angular_velocity[,r=reward][,d=done][,e=episode].
0,-0.015,0.018,0.038,-0.026,e=0*\n*
1,-0.014,0.213,0.037,-0.307,r=1*\n*
0,-0.010,0.018,0.031,-0.039,r=1,d=1*\n*
```

Packed in order: group prefix, then each step. `head_output_indices` points at every `*\n*` token.


In [ ]:
backbone = TransformerBackbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
    pretrained=PRETRAINED,
)

head = RegressionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
    use_norm=True,
    propagate_gradient=1.0,
)

model = Model(
    backbone=backbone,
    heads={"action_value": head},
    action_source="action_value",
    reasoner=None,
).train().to(device)
print(model)


## Training Phase

Same one-step DQN loop as `02`. `boundary_discount` multiplies `gamma_step` on every link, then the episode and task extras (`1.0` when that code is `0`). Episode extras stay `1.0`, so a fall does not cut the return: later episodes in the task still count. Task extras are `0.0`, so the return stops at the task boundary. `temperature=0` is hard max-Q. `double=False` bootstraps from delayed Q. `gate=None` is the one-step target. `cross_group_backups="bootstrap"` fills `V` on the step before a group boundary.

`delayed_model = model.copy(heads=True, backbone=True, reasoner=False)` copies the heads and the backbone. `model_polyak` moves those copies toward the online weights after each optimizer step.


In [ ]:
optimizer = AdamW(
    params=[
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
        {"params": model.heads.parameters(), "lr": LR_HEADS},
    ],
    lr=LR_BACKBONE,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
delayed_model = model.copy(heads=True, backbone=True, reasoner=False)
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_episode_terminal=1.0,
        gamma_episode_truncated=1.0,
        gamma_task_terminal=0.0,
        gamma_task_truncated=0.0,
    ),
    temperature=0.0,
    double=False,
    cross_group_backups="bootstrap",
    gate=None,
)

def run_train(*, model: Model, delayed_model: Model, optimizer: AdamW, objective: DqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data, group_id = loader.next_batch()
        out = model(inputs)
        with torch.no_grad():
            delayed_out = delayed_model(inputs)
        losses, metrics = objective(
            objective_data=to_device(data=objective_data, device=device),
            group_id=group_id.to(device),
            predictions=out.predictions["action_value"],
            delayed_predictions=delayed_out.predictions["action_value"],
            reward_center=None,
            delayed_reward_center=None,
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model_polyak(
            online=model,
            delayed=delayed_model,
            tau_heads=POLYAK_TAU_HEADS,
            tau_backbone=POLYAK_TAU_BACKBONE,
        )
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to `TOKENIZER_ID`. `input_index` is part of that spec. `load_model` rebuilds the model; `load_tokenizer` on the tokenizer repo rebuilds the fields.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(
    model=model,
    tokenizer=tokenizer,
    repo_id=MODEL_ID,
    tokenizer_repo_id=TOKENIZER_ID,
    private=False,
    clear=True,
)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")
